In [6]:
import pandas as pd

# Load population dataset
pop_file = "Population Figures & Projections.csv"
df_population = pd.read_csv(pop_file)

# Load country groupings dataset
group_file = "Country Groupings.xlsx"
df_country_groupings = pd.read_excel(group_file, sheet_name="List of economies")

df_population.head(), df_country_groupings.head()

(  Country Name Country Code  Time Time Code  \
 0  Afghanistan          AFG  1960    YR1960   
 1  Afghanistan          AFG  1961    YR1961   
 2  Afghanistan          AFG  1962    YR1962   
 3  Afghanistan          AFG  1963    YR1963   
 4  Afghanistan          AFG  1964    YR1964   
 
    Population, female [SP.POP.TOTL.FE.IN]  \
 0                               4145945.0   
 1                               4233771.0   
 2                               4326881.0   
 3                               4424511.0   
 4                               4526691.0   
 
    Population, male [SP.POP.TOTL.MA.IN]  \
 0                             4476521.0   
 1                             4556369.0   
 2                             4642166.0   
 3                             4732954.0   
 4                             4828823.0   
 
    Population growth (annual %) [SP.POP.GROW]  Rural population [SP.RUR.TOTL]  \
 0                                         NaN                       7898093.0   
 1

In [2]:
# Select required columns and rename them
pop_cols_map = {
    'Country Code': 'Code',
    'Country Name': 'Country',
    'Time': 'Year',
    'Rural population [SP.RUR.TOTL]': 'Rural Population',
    'Urban population [SP.URB.TOTL]': 'Urban Population'
}

df_pop = df_population[list(pop_cols_map.keys())].rename(columns=pop_cols_map)
df_pop.head()

,Code,Country,Year,Rural Population,Urban Population
0,AFG,Afghanistan,1960,7898093.0,724373.0
1,AFG,Afghanistan,1961,8026804.0,763336.0
2,AFG,Afghanistan,1962,8163985.0,805062.0
3,AFG,Afghanistan,1963,8308019.0,849446.0
4,AFG,Afghanistan,1964,8458694.0,896820.0


In [3]:
# Convert to numeric, coerce non-numeric values to NaN
df_pop['Rural Population'] = pd.to_numeric(df_pop['Rural Population'], errors='coerce')
df_pop['Urban Population'] = pd.to_numeric(df_pop['Urban Population'], errors='coerce')

df_pop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19747 entries, 0 to 19746
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Code              19747 non-null  object 
 1   Country           19747 non-null  object 
 2   Year              19747 non-null  int64  
 3   Rural Population  19454 non-null  float64
 4   Urban Population  19454 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 771.5+ KB


In [4]:
# Creating Total Population
df_pop['Total Population'] = df_pop['Urban Population'] + df_pop['Rural Population']

df_pop[['Urban Population', 'Rural Population', 'Total Population']].head()

,Urban Population,Rural Population,Total Population
0,724373.0,7898093.0,8622466.0
1,763336.0,8026804.0,8790140.0
2,805062.0,8163985.0,8969047.0
3,849446.0,8308019.0,9157465.0
4,896820.0,8458694.0,9355514.0


In [5]:
# Select and rename grouping columns
group_cols = ['Code', 'Region', 'Income group']
df_group = df_country_groupings[group_cols].rename(columns={'Income group': 'Income Group'})

df_group.head()

,Code,Region,Income Group
0,AFG,South Asia,Low income
1,ALB,Europe & Central Asia,Upper middle income
2,DZA,Middle East & North Africa,Upper middle income
3,ASM,East Asia & Pacific,High income
4,AND,Europe & Central Asia,High income


In [6]:
# Left join population with groupings
df_merged = pd.merge(df_pop, df_group, on='Code', how='left')

# Replace missing grouping values
df_merged['Region'] = df_merged['Region'].fillna('Aggregates')
df_merged['Income Group'] = df_merged['Income Group'].fillna('Aggregates')

df_merged.head()

,Code,Country,Year,Rural Population,Urban Population,Total Population,Region,Income Group
0,AFG,Afghanistan,1960,7898093.0,724373.0,8622466.0,South Asia,Low income
1,AFG,Afghanistan,1961,8026804.0,763336.0,8790140.0,South Asia,Low income
2,AFG,Afghanistan,1962,8163985.0,805062.0,8969047.0,South Asia,Low income
3,AFG,Afghanistan,1963,8308019.0,849446.0,9157465.0,South Asia,Low income
4,AFG,Afghanistan,1964,8458694.0,896820.0,9355514.0,South Asia,Low income


In [7]:
# Calculate Urbanisation Rate
df_merged['Urbanization Rate'] = (
    df_merged['Urban Population'] / df_merged['Total Population']
).fillna(0)

df_merged[['Urban Population', 'Total Population', 'Urbanization Rate']].head()

,Urban Population,Total Population,Urbanization Rate
0,724373.0,8622466.0,0.08401
1,763336.0,8790140.0,0.08684
2,805062.0,8969047.0,0.08976
3,849446.0,9157465.0,0.09276
4,896820.0,9355514.0,0.09586


In [8]:
# Filter dataset to required years
df_final = df_merged[(df_merged['Year'] >= 1960) & (df_merged['Year'] <= 2050)]

df_final.head()

,Code,Country,Year,Rural Population,Urban Population,Total Population,Region,Income Group,Urbanization Rate
0,AFG,Afghanistan,1960,7898093.0,724373.0,8622466.0,South Asia,Low income,0.08401
1,AFG,Afghanistan,1961,8026804.0,763336.0,8790140.0,South Asia,Low income,0.08684
2,AFG,Afghanistan,1962,8163985.0,805062.0,8969047.0,South Asia,Low income,0.08976
3,AFG,Afghanistan,1963,8308019.0,849446.0,9157465.0,South Asia,Low income,0.09276
4,AFG,Afghanistan,1964,8458694.0,896820.0,9355514.0,South Asia,Low income,0.09586


In [11]:
output_filename = "master_population_data_clean.csv"
df_final.to_csv(output_filename, index=False)

print( output_filename)

master_population_data_clean.csv


In [10]:
df_final.sample(10)

,Code,Country,Year,Rural Population,Urban Population,Total Population,Region,Income Group,Urbanization Rate
1541,BRB,Barbados,2045,168016.0,105844.0,273860.0,Latin America & Caribbean,High income,0.386489
3726,CHL,Chile,2046,1807358.0,18881267.0,20688625.0,Latin America & Caribbean,High income,0.912640
9511,KOR,"Korea, Rep.",2007,8942697.0,39740941.0,48683638.0,East Asia & Pacific,High income,0.816310
12861,NZL,New Zealand,1990,508061.0,2821739.0,3329800.0,East Asia & Pacific,High income,0.847420
17488,THA,Thailand,1976,31677548.0,10204580.0,41882128.0,East Asia & Pacific,Upper middle income,0.243650
17229,SYR,Syrian Arab Republic,1990,6337150.0,6071846.0,12408996.0,Middle East & North Africa,Low income,0.489310
14663,RUS,Russian Federation,1972,47384351.0,84524649.0,131909000.0,Europe & Central Asia,High income,0.640780
15123,SAU,Saudi Arabia,1977,3377676.0,5377543.0,8755219.0,Middle East & North Africa,High income,0.614210
1725,BEL,Belgium,2047,142371.0,12005320.0,12147691.0,Europe & Central Asia,High income,0.988280
9072,JOR,Jordan,2023,904697.0,10432355.0,11337052.0,Middle East & North Africa,Lower middle income,0.920200
